### Tools

Models can request to call tools that perform tasks such as fetching data from a database, searching the web, or running code. Tools are pairings of:

1. A schema, including the name of the tool, a description, and/or argument definitions (often a JSON schema)
2. A function or coroutine to execute.

In [ ]:
import os
from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

model = init_chat_model("groq:qwen/qwen3-32b")
response = model.invoke("Why do parrots talk?")
response
# 01:07:00

AIMessage(content='<think>\nOkay, so I need to figure out why parrots talk. Let me start by recalling what I know about parrots. They\'re birds, right? And some species are known for mimicking human speech. But why do they do that? Maybe it\'s related to their natural behavior. Let me think.\n\nIn the wild, parrots live in social groups. They might use vocalizations to communicate with each other. So maybe talking is a way to interact with their flock. But when they\'re domesticated, they talk to humans. Is it similar to how they communicate with other parrots? Also, parrots have a syrinx, which is their vocal organ, allowing them to produce a wide range of sounds. That\'s probably why they can mimic human speech.\n\nI remember reading that parrots don\'t just repeat words; they can associate words with meanings. So it\'s not just mimicry but some level of understanding. But why would they evolve this ability? Maybe in the wild, they use it to identify individuals, find mates, or warn 

In [2]:
from langchain.tools import tool


@tool  # By calling @tool we are creating a tool
def get_weather(location: str) -> str:
    """get weather at a location"""
    return f"It's sunny in {location}"


model_with_tools = model.bind_tools([get_weather])

In [4]:
response = model_with_tools.invoke("What is the weather in Kolkata")
print(response)
for tool_call in response.tool_calls:
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")

content='' additional_kwargs={'reasoning_content': 'Okay, the user is asking for the weather in Kolkata. Let me check the tools available. There\'s a function called get_weather that takes a location parameter. Since Kolkata is the location specified, I need to call that function with "Kolkata" as the argument. I\'ll make sure the JSON is correctly formatted with the name and arguments. No other tools are available, so this should be straightforward. Let me double-check the spelling of the city to ensure it\'s correct. Yep, Kolkata is the right name. Alright, time to structure the tool call properly within the XML tags.\n', 'tool_calls': [{'id': '2nxph4ptv', 'function': {'arguments': '{"location":"Kolkata"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 143, 'prompt_tokens': 151, 'total_tokens': 294, 'completion_time': 0.251080793, 'completion_tokens_details': {'reasoning_tokens': 118}, 'prompt_time': 0.005968147, 'prompt_tokens_

### Tool Execution Loops

In [6]:
# Step 1: Model generates tool calls
messages = [{"role": "user", "content": "What's the weather in Kolkata"}]
ai_message = model_with_tools.invoke(messages)
messages.append(ai_message)

# Step 2: Executing tools & collect results
for tool_call in ai_message.tool_calls:
    # Execute the tool with the generated arguments
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

# Step 3: Pass the result to the model for final response
final_response = model_with_tools.invoke(messages)
print(final_response.text)

The weather in Kolkata is sunny.


In [7]:
messages # To show the full message(response)

[{'role': 'user', 'content': "What's the weather in Kolkata"},
 AIMessage(content='', additional_kwargs={'reasoning_content': "Okay, the user is asking for the weather in Kolkata. I need to use the get_weather function. The function requires a location parameter. Kolkata is a city in India, so I should check if that's the correct location. I don't see any other parameters needed. Just need to call get_weather with location set to Kolkata.\n", 'tool_calls': [{'id': 'nnv6zz3rm', 'function': {'arguments': '{"location":"Kolkata"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 92, 'prompt_tokens': 151, 'total_tokens': 243, 'completion_time': 0.167166297, 'completion_tokens_details': {'reasoning_tokens': 67}, 'prompt_time': 0.006696146, 'prompt_tokens_details': None, 'queue_time': 0.055936653, 'total_time': 0.173862443}, 'model_name': 'qwen/qwen3-32b', 'system_fingerprint': 'fp_5cf921caa2', 'service_tier': 'on_demand', 'finish_reason'